# 04 - Generalized coupled mode equations

This tutorial shows how to simulate the nonlinear response of a TWPA with extended sets of modes, going beyond the basic pump-signal-idler system to include pump harmonics, frequency conversion processes and higher-order effects, and how to account for reflections and the connectors at the device ends.

### Overview

The basic 3-wave mixing (3WM) system is a good starting point, but real devices also show:

* **Pump harmonics**: 2nd, 3rd and higher harmonics of the pump
* **Frequency conversion**: processes such as pump + signal, pump + idler
* **Signal/idler harmonics**: higher harmonics of signal and idler
* **Device imperfections**: losses along the line and reflections at its ends

`twpasolver` describes the modes with `ModeSet` objects and solves the resulting CMEs with two models, `simple` and `bloch`.

### Initialize TWPAnalysis

We load a KITWPA model and plot its response. Since higher-order processes are involved, the frequency grid extends well above the pump to cover its harmonics.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

from twpasolver import ModeSet, Ports, TWPAnalysis
from twpasolver.cmes.solver_config import BlochSolverConfig
from twpasolver.logger import log
from twpasolver.models import Inductance

log.setLevel(logging.WARNING)

plt.rcParams["font.size"] = 13.5
plt.rcParams["axes.axisbelow"] = True

twpa_file = "imskit_v3_holotype.json"  # Representative circuit model from https://doi.org/10.1103/yg71-j2dn
a = TWPAnalysis(twpa=twpa_file, freqs=(0.2, 60, 2e-3), unit="GHz")

pump = 16.2
ax = a.plot_response(pump_freq=pump)
for i in range(2):
    ax[i].axvline(2 * pump, ls="--", c="r")
    ax[i].axvline(3 * pump, ls="--", c="r")
ax[0].set_ylim(-20, 1)

The grid extends to 60 GHz to include the 2nd and 3rd pump harmonics (red dashed lines). Because the line is dispersive, the harmonics have their own propagation constants and take part in further mixing processes.

## Mode sets

Describing the nonlinear response properly requires more modes than pump, signal and idler. A `ModeSet` holds an arbitrary number of modes connected by frequency relations:

* **Independent and dependent modes**: modes not defined by any relation are the independent tones (here pump `p` and signal `s`); all others follow from them
* **Frequencies**: the frequency of every mode is an integer combination of the independent tones, computed once and applied to whole frequency arrays
* **RWA term selection**: the 3WM and 4WM terms that survive the rotating wave approximation (RWA) are found from the relations, and the analysis uses them to build the CMEs

When a gain is computed, the physical parameters of each mode (wavenumber, losses, impedance) are interpolated from the linear response at the mode frequencies.

Here is a custom mode set with the idler and the 2nd pump harmonic:

In [ ]:
modes_p2 = ModeSet(["p", "s", "i", "p2"], [["i", "p-s"], ["p2", "p+p"]])
print(modes_p2)
_ = modes_p2.plot_relations(layout="circular")

The graph shows the relations between modes:
* **Green nodes**: independent modes (pump `p` and signal `s`)
* **Blue nodes**: dependent modes (idler `i` and 2nd pump harmonic `p2`)
* **Blue arrows**: positive frequency contributions
* **Red dashed arrows**: negative frequency contributions

`rwa_terms(power)` lists the RWA terms with `power` mode factors (2 for 3WM, 3 for 4WM). Each tuple is `(driven mode index, factor indices, driven mode, factors, coefficient)`; factor indices at or above the number of modes, printed with a minus sign, are complex conjugates, and the coefficient $1/\prod n_r!$ accounts for repeated factors. The 3WM terms here come from $\omega_s+\omega_i=\omega_p$ and $\omega_p+\omega_p=\omega_{p2}$:

In [ ]:
for term in modes_p2.rwa_terms(2):
    print(term)

For realistic simulations one should include pump harmonics, frequency conversion and signal/idler harmonics. `ModeSet.extended_3wm` builds such a set automatically:

In [ ]:
modes_ext = ModeSet.extended_3wm(
    n_pump_harmonics=2, n_frequency_conversion=2, n_signal_harmonics=1
)
print(modes_ext)
print(
    f"{len(modes_ext.rwa_terms(2))} 3WM terms, {len(modes_ext.rwa_terms(3))} 4WM terms"
)
_ = modes_ext.plot_relations()

This extended set includes:
* **Pump harmonics**: `p`, `p2`, `p3` (fundamental, 2nd and 3rd harmonic)
* **Basic 3WM**: `s`, `i` (signal and idler)
* **Frequency conversion**: `ps` (p+s), `pi` (p+i), and higher-order processes generated by the pump harmonics
* **Signal harmonics**: `s2`, `i2` (2nd harmonics of signal and idler)

The hierarchical layout puts the independent modes at the top and the derived modes below them. The number of RWA terms grows quickly with the number of modes, and with it the cost of a solve.

## Gain with different models

The `model` argument of `gain` selects the CME solver:

* **`simple`** (default): the textbook coupled mode equations of a homogeneous line, with forward-propagating waves only. The periodic loading enters only through the Bloch wavenumber, and the couplings are those of a uniform medium, so they are approximate for loaded lines (especially close to stopbands). Reflections and connectors are ignored. It is fast.
* **`bloch`**: the forward-backward model in the Bloch-mode basis. The amplitudes are those of the exact forward and backward Bloch modes of the supercell, the couplings include the current profile inside the supercell, and the boundary conditions at the ports are exact. It solves self-consistently for forward and backward waves (configured by a `BlochSolverConfig`) and treats modes close to a stopband edge on a fixed basis. It is more accurate and more expensive.

Both models take the same `modes` argument, so we first compare the basic 3WM set with the extended one using the fast `simple` model. `gain` returns a `GainResult`, whose fields are attributes (e.g. `r.gain_db`) and which plots itself with `r.plot()`:

In [ ]:
s_arange = np.arange(2, 13, 0.1)

res_base = a.gain(signal_freqs=s_arange, Is0=1e-7, pump=pump)
res_extended = a.gain(
    signal_freqs=s_arange,
    Is0=1e-7,
    pump=pump,
    model="simple",
    modes=modes_ext,
)

ax = res_base.plot()
ax.plot(s_arange, res_extended.gain_db)
ax.legend(["basic 3WM", "extended, simple"], bbox_to_anchor=(1.1, 0.6))

The extra modes exchange power with pump, signal and idler and reshape the gain profile. Even with many more modes, the `simple` solver stays fast thanks to Numba acceleration, once compiled.

## Adding reflections: the `bloch` model

The `bloch` model includes reflections at the ends of the line and the exact Bloch-mode couplings, solving forward and backward waves self-consistently. It is configured by `BlochSolverConfig`:

* `passes`: maximum number of forward-backward iterations
* `convergence_threshold`: relative residual below which the iteration is considered converged
* `anderson_depth`: number of previous passes used to accelerate the iteration; when raising `passes`, raise it too (for example to `passes - 10`)
* `kappa_cutoff`: phase-mismatch cutoff used to prune negligible 3WM/4WM terms, which can greatly speed up the computation
* `m_max`: largest reciprocal lattice vector of the supercell used for quasi-phase-matched terms
* `edge_threshold`, `simulate_stopband_modes`: treatment of the modes close to a stopband edge
* `save_all_passes`: if `True`, keep every intermediate pass so the convergence can be inspected

Both models integrate the CMEs with an adaptive RK45 solver configured by `RKSolverConfig` (`atol`, `rtol`, `first_step`, `max_steps_per_cell`), passed to `gain` as `rk_config`.

A `GainResult` has the same fields for both models, but the `bloch` model fills more of them: besides `gain_db` (S21) it gives the reflection `gain_db_11` (S11), the backward currents `I_bwd` and the solver tables in `diagnostics`.

In [ ]:
bloch_config = BlochSolverConfig(
    passes=40,
    anderson_depth=30,
    kappa_cutoff=0.25,
    convergence_threshold=1e-2,
    save_all_passes=False,
)

res_bloch = a.gain(
    signal_freqs=s_arange,
    Is0=1e-7,
    pump=pump,
    model="bloch",
    modes=modes_ext,
    bloch_config=bloch_config,
)
print(f"converged at {res_bloch.converged.sum()} of {len(s_arange)} signal frequencies")

ax = res_bloch.plot()
ax.plot(s_arange, res_extended.gain_db, ls="--")
ax.plot(s_arange, res_bloch.gain_db_11, ls="-.")
ax.legend(["bloch, S21", "simple", "bloch, S11"], bbox_to_anchor=(1.1, 0.6))

The `bloch` model shows ripples in the gain profile caused by reflections, which the `simple` model cannot capture by construction, and its exact couplings also shift the gain itself. For quantitative comparison with experiment use this model, at the cost of extra computation (typically tens of times slower than `simple`, depending on `passes` and `kappa_cutoff`).

### Port currents and S-parameters

The gains are ratios of the currents in the external lines at the two ends of the device, stored with shape `(n_freq, n_modes)`:

* `a_in`, `a_out`: currents incident on the device from the line at $x=0$ and at $x=L$
* `b_in`, `b_out`: currents leaving the device into the line at $x=0$ and at $x=L$

For the signal mode, $S_{21}=b_\mathrm{out}/a_\mathrm{in}$, $S_{11}=b_\mathrm{in}/a_\mathrm{in}$, $S_{12}=b_\mathrm{in}/a_\mathrm{out}$ and $S_{22}=b_\mathrm{out}/a_\mathrm{out}$ (`gain_db`, `gain_db_11`, `gain_db_12`, `gain_db_22`, in dB). Here only $x=0$ is driven, so $a_\mathrm{out}=0$ and S12, S22 are NaN; driving from $x=L$ requires `initial_amplitudes_bwd`. `r.mode(label)` gives the index of a mode on the mode axis:

In [ ]:
s = res_bloch.mode("s")
S11 = 20 * np.log10(np.abs(res_bloch.b_in[:, s] / res_bloch.a_in[:, s]))
print("a_in shape:", res_bloch.a_in.shape, " modes:", list(res_bloch.mode_labels))
print(
    "S11 from port currents matches gain_db_11:", np.allclose(S11, res_bloch.gain_db_11)
)
print(
    "S12, S22 all NaN:",
    np.isnan(res_bloch.gain_db_12).all(),
    np.isnan(res_bloch.gain_db_22).all(),
)
p2 = res_bloch.mode("p2")
print(
    f"max |b_out| of p2: {np.abs(res_bloch.b_out[:, p2]).max():.2e} A, max |b_in| of p2: {np.abs(res_bloch.b_in[:, p2]).max():.2e} A"
)

The outgoing currents exist for every mode, so `b_in` and `b_out` also show how much of each generated mode (here the 2nd pump harmonic) leaves the device at either end.

## Mode currents along the line

`r.plot_currents()` plots the current of each mode, averaged over the signal frequencies, as a function of position along the line (in unit cells). This helps identify which processes matter and how energy flows between modes:

In [ ]:
_ = res_bloch.plot_currents()

The `bloch` model also gives the backward-propagating wave. `r.I_bwd` is stored on the same physical position grid `r.x` as `r.I_fwd` ($x=0$ at the input port), so `r.I_bwd[..., 0]` is the backward wave arriving at the input and `r.I_bwd[..., -1]` the one just launched at the output end:

In [ ]:
print(
    "x:",
    res_bloch.x.shape,
    " I_fwd:",
    res_bloch.I_fwd.shape,
    " I_bwd:",
    res_bloch.I_bwd.shape,
)
_ = res_bloch.plot_currents(backward=True)

The backward modes evolve differently from the forward ones, since they take part in different nonlinear processes. The current plots show:
* **Pump depletion**: the pump current decreases along the line as energy is transferred to the other modes or dissipated in a lossy device
* **Signal amplification**: the signal current grows in the gain region
* **Idler generation**: the idler current increases as the parametric process generates idler photons
* **Harmonic contributions**: the relative importance of the higher-order modes
* **Backward wave**: nonzero because of reflections at the ports and along the line; it feeds back into the forward wave and is the origin of the gain ripples seen above

Modes with negligible current throughout the device can often be dropped from the mode set to save computation.

### Zooming in on the ripples

We zoom in on a narrow frequency range where the effect of reflections on the gain is most visible, saving every forward-backward pass:

In [ ]:
bloch_config.save_all_passes = True
s_arange_narrow = np.arange(6.1, 6.3, 0.002)
res_narrow = a.gain(
    signal_freqs=s_arange_narrow,
    Is0=1e-7,
    pump=pump,
    model="bloch",
    modes=modes_ext,
    bloch_config=bloch_config,
)
ax = res_narrow.plot()

### Forward-backward convergence

Because the `bloch` model iterates forward and backward waves until self-consistency, it is useful to check how quickly the gain converges with the number of passes. With `save_all_passes=True`, `r.history` holds every pass with a leading pass axis (`r.history["gain_db"][p]` is S21 after pass `p`). The pass axis has length `passes`; a point that converges earlier is NaN after its last pass, and `r.diagnostics["passes_used"]` gives the number of passes each point ran.

In [ ]:
passes_gain_db = res_narrow.history["gain_db"]
n_passes = len(passes_gain_db)
print(
    f"{n_passes} passes stored, converged at {res_narrow.converged.sum()} of {len(s_arange_narrow)} frequencies"
)

fig, ax = plt.subplots(figsize=(7, 4.5))
for i, g in enumerate(passes_gain_db):
    ax.plot(
        s_arange_narrow,
        g,
        color=plt.cm.viridis(i / max(1, n_passes - 1)),
        lw=1,
        label=f"pass {i + 1}" if i in (0, n_passes - 1) else None,
    )
ax.plot(
    s_arange_narrow, res_narrow.gain_db, color="k", lw=2, ls="--", label="converged"
)
ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel("Gain [dB]")
ax.set_title("Forward-backward convergence")
ax.legend()
ax.grid(alpha=0.3)

Most of the change happens in the first passes: the first pass already gives the smooth gain, and the following ones add the ripples from the backward wave. `convergence_threshold` sets when the iteration stops, and `r.converged` flags the points that reached it within `passes`.

### Modeling external components

TWPAs are very sensitive to reflections, so reproducing the ripple amplitude requires modeling what the device is connected to. Connectors are `TwoPortModel`s passed as `ports=Ports(input=..., output=...)`: port 1 of a connector faces the external line (impedance `Ports.Z_ext`, 50 Ohm by default) and port 2 the TWPA, and `input` is used at both ends when `output` is not given. Here a small series inductance models the wirebonds connecting the sample to the external lines. Connectors only enter the `bloch` model.

In [ ]:
wirebonds = Inductance(L=2e-10)
a.ports = Ports(input=wirebonds)

bloch_config.save_all_passes = False
res_narrow_wb = a.gain(
    signal_freqs=s_arange_narrow,
    Is0=1e-7,
    pump=pump,
    model="bloch",
    modes=modes_ext,
    bloch_config=bloch_config,
)
ax = res_narrow.plot()
ax.plot(s_arange_narrow, res_narrow_wb.gain_db)
ax.legend(["Matched to 50 Ohm", "With wirebonds"], bbox_to_anchor=(1.1, 0.6))

The wirebond inductance adds a reactive mismatch at both ends: here the ripples grow from about 1 dB to 6-7 dB peak to peak around the same mean gain, showing why the connectors must be modeled to compare ripple amplitudes with measurements.

Devices with a position-dependent design, such as the Floquet TWPA, and resonantly phase-matched devices are covered in notebook 05.